In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

In [0]:
bronze_df = spark.read.table(f'adb_caren.bronze.{table_name}').filter(f"landing_timestamp = '{landing_timestamp}'")

In [0]:
from pyspark.sql import functions as F
bronze_df = spark.read.table(f'adb_caren.bronze.{table_name}').filter(F.col("landing_timestamp") == F.to_timestamp(F.lit(landing_timestamp), "yyyy-MM-dd_HH:mm:ss"))
display(bronze_df)

In [0]:
from pyspark.sql import functions as F
updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn('update_timestamp', F.current_timestamp())

In [0]:
display(updated_df)

In [0]:
from delta.tables import DeltaTable

target_table = f'adb_caren.silver.{table_name}'
# Default to FULL when load_type is not provided (preserves original behaviour)
load_type_normalized = (load_type or 'FULL').strip().upper()

if load_type_normalized == 'FULL':
    # Full load: overwrite the target table
    updated_df.write.mode('overwrite').saveAsTable(target_table)

elif load_type_normalized == 'APPEND':
    # Append: add the incoming records to the target table
    updated_df.write.mode('append').saveAsTable(target_table)

elif load_type_normalized == 'MERGE':
    # Merge (upsert): update existing records and insert new ones based on merge keys
    keys = [k.strip() for k in merge_keys.split(',') if k.strip()]
    if not keys:
        raise ValueError("merge_keys must be provided for MERGE load_type (comma-separated column names)")
    missing_keys = [k for k in keys if k not in updated_df.columns]
    if missing_keys:
        raise ValueError(f"merge_keys not found in source data: {missing_keys}")

    if not spark.catalog.tableExists(target_table):
        # First load: target does not exist yet, create it
        updated_df.write.mode('overwrite').saveAsTable(target_table)
    else:
        # Avoid multiple source rows matching the same target row
        source_df = updated_df.dropDuplicates(keys)
        merge_condition = ' AND '.join([f"t.`{k}` = s.`{k}`" for k in keys])
        # Keep the original insert_timestamp on updates; refresh everything else
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != 'insert_timestamp'}
        insert_values = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        (DeltaTable.forName(spark, target_table).alias('t')
            .merge(source_df.alias('s'), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_values)
            .execute())

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

print(f"{load_type_normalized} load completed for {target_table}")

In [0]:
%sql
select * from adb_caren.silver.hospitals

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))

In [0]:
from delta.tables import DeltaTable

target_table = f'adb_caren.silver.{table_name}'
# Default to FULL when load_type is not provided (preserves original behaviour)
load_type_normalized = (load_type or 'FULL').strip().upper()

print(f"="*60)
print(f"🚀 Starting Silver Load")
print(f"="*60)
print(f"📋 Table Name       : {table_name}")
print(f"🎯 Target Table     : {target_table}")
print(f"📦 Load Type        : {load_type_normalized}")
print(f"🔑 Merge Keys       : {merge_keys if merge_keys else 'N/A'}")
print(f"🕐 Landing Timestamp: {landing_timestamp}")
print(f"📊 Source Row Count  : {updated_df.count()}")
print(f"📄 Source Columns   : {updated_df.columns}")
print(f"="*60)

if load_type_normalized == 'FULL':
    # Full load: overwrite the target table
    print(f"\n⚙️  [FULL LOAD] Overwriting entire target table...")
    print(f"   Query: updated_df.write.mode('overwrite').saveAsTable('{target_table}')")
    updated_df.write.mode('overwrite').saveAsTable(target_table)
    print(f"   ✅ Full overwrite completed successfully.")

elif load_type_normalized == 'APPEND':
    # Append: add the incoming records to the target table
    print(f"\n⚙️  [APPEND LOAD] Appending records to target table...")
    print(f"   Query: updated_df.write.mode('append').saveAsTable('{target_table}')")
    updated_df.write.mode('append').saveAsTable(target_table)
    print(f"   ✅ Append completed successfully.")

elif load_type_normalized == 'MERGE':
    # Merge (upsert): update existing records and insert new ones based on merge keys
    keys = [k.strip() for k in merge_keys.split(',') if k.strip()]
    print(f"\n⚙️  [MERGE LOAD] Performing upsert into target table...")
    print(f"   Parsed Merge Keys: {keys}")

    if not keys:
        raise ValueError("merge_keys must be provided for MERGE load_type (comma-separated column names)")
    missing_keys = [k for k in keys if k not in updated_df.columns]
    if missing_keys:
        raise ValueError(f"merge_keys not found in source data: {missing_keys}")

    if not spark.catalog.tableExists(target_table):
        # First load: target does not exist yet, create it
        print(f"   ⚠️  Target table '{target_table}' does not exist. Creating with initial data...")
        print(f"   Query: updated_df.write.mode('overwrite').saveAsTable('{target_table}')")
        updated_df.write.mode('overwrite').saveAsTable(target_table)
        print(f"   ✅ Initial table created successfully.")
    else:
        print(f"   ✅ Target table '{target_table}' exists. Proceeding with MERGE...")
        # Avoid multiple source rows matching the same target row
        source_df = updated_df.dropDuplicates(keys)
        dedup_count = source_df.count()
        print(f"   📊 Source rows after deduplication on keys: {dedup_count}")

        merge_condition = ' AND '.join([f"t.`{k}` = s.`{k}`" for k in keys])
        print(f"\n   🔗 Merge Condition:")
        print(f"      {merge_condition}")

        # Keep the original insert_timestamp on updates; refresh everything else
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != 'insert_timestamp'}
        insert_values = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        print(f"\n   📝 WHEN MATCHED UPDATE SET (excludes insert_timestamp):")
        for col, val in update_set.items():
            print(f"      {col} = {val}")

        print(f"\n   📝 WHEN NOT MATCHED INSERT VALUES:")
        for col, val in insert_values.items():
            print(f"      {col} = {val}")

        print(f"\n   🔄 Executing MERGE...")
        (DeltaTable.forName(spark, target_table).alias('t')
            .merge(source_df.alias('s'), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_values)
            .execute())
        print(f"   ✅ MERGE completed successfully.")

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

# Final summary
result_df = spark.read.table(target_table)
result_count = result_df.count()
print(f"\n{'='*60}")
print(f"✅ {load_type_normalized} load completed for {target_table}")
print(f"📊 Total rows in target table after load: {result_count}")
print(f"{'='*60}")